# Walkthrough — strategies × models × datasets

This notebook reads the bucket tables and summary JSONs produced by
`scripts/run_metric_runner.py` (called by `Workflow/run_full.py`) and
renders the headline visualisations.

Pre-requisite:

```bash
PYTHONPATH=. python Workflow/run_full.py
```

The metric used is **metrik-4** from
[`domain-model-metrics`](https://github.com/VasiliySeibert/domainModel-Metrics-Comparison).
Score buckets are `[0, 0.1)` / `[0.1, 0.2)` / `[0.2, 0.3)` / `[0.3, 1.0]`.
Rows in every table are `(strategy, model)` pairs — model is `-` for
the rule-based heuristic.

In [ ]:
from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

RESULTS = Path('Workflow/Results')
assert RESULTS.is_dir(), 'Run Workflow/run_full.py first'

summary = pd.read_csv(RESULTS / '_summary.csv')
errors  = pd.read_csv(RESULTS / '_errors.csv') if (RESULTS / '_errors.csv').exists() else None
print(f'{len(summary)} (strategy, model, dataset, element) rows')
print(f'{len(errors) if errors is not None else 0} failure rows')
summary.head()

## 1. Bucket tables (the headline deliverable)

One CSV per `(dataset, element)`. Six files total:
    Workflow/Results/_bucket_<dataset>_class_score.csv
    Workflow/Results/_bucket_<dataset>_attribute_score.csv
    Workflow/Results/_bucket_<dataset>_association_score.csv

Each row is one `(strategy, model)` pair. Columns are bucket counts
("[0, 0.1)", "[0.1, 0.2)", "[0.2, 0.3)", "[0.3, 1.0]") plus `n`,
`n_failed`, `mean`, `median`. Read it as: *"of the N records this
strategy+model produced, K1 fell in bucket 1, K2 in bucket 2, ..."*.

In [ ]:
bucket_files = sorted(RESULTS.glob('_bucket_*.csv'))
for bf in bucket_files:
    print(f'\n=== {bf.name} ===')
    display(pd.read_csv(bf))

## 2. Heatmap of bucket distribution (Class element)

Visual version of the bucket table: each row is `(strategy, model)`,
each column is a bucket. The intensity in cell (r, c) shows how many
of the N records for that (strategy, model) landed in bucket c.

In [ ]:
def plot_bucket_heatmap(dataset: str, element: str):
    f = RESULTS / f'_bucket_{dataset}_{element}.csv'
    df = pd.read_csv(f)
    bucket_cols = ['[0, 0.1)', '[0.1, 0.2)', '[0.2, 0.3)', '[0.3, 1.0]']
    df['row'] = df['strategy'] + ' / ' + df['model']
    matrix = df.set_index('row')[bucket_cols]

    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(matrix))))
    im = ax.imshow(matrix.values, aspect='auto', cmap='YlOrRd')
    ax.set_xticks(range(len(bucket_cols)))
    ax.set_xticklabels(bucket_cols, rotation=20, ha='right')
    ax.set_yticks(range(len(matrix.index)))
    ax.set_yticklabels(matrix.index)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            v = matrix.values[i, j]
            if v:
                ax.text(j, i, str(v), ha='center', va='center', fontsize=8, color='black')
    ax.set_title(f'{dataset} — {element} (records per bucket)')
    plt.colorbar(im, ax=ax, label='count')
    plt.tight_layout()
    plt.show()

for ds in ('kaiser', 'reference'):
    plot_bucket_heatmap(ds, 'class_score')

## 3. Mean score per (strategy, model)

Headline single-number comparison. Higher is better. Same colour
scheme as the bucket heatmap (yellow = best, red = worst).

In [ ]:
def plot_mean_bars(dataset: str):
    sub = summary[summary['dataset'] == dataset]
    fig, axes = plt.subplots(1, 3, figsize=(16, max(3, 0.4 * sub['strategy'].nunique())), sharey=False)
    for ax, element in zip(axes, ['class_score', 'attribute_score', 'association_score']):
        pivot = (sub[sub['element'] == element]
                 .pivot_table(index='strategy', columns='model', values='mean'))
        im = ax.imshow(pivot.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index)
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                v = pivot.values[i, j]
                if pd.notna(v):
                    ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8)
        ax.set_title(element)
    fig.suptitle(f'{dataset} — mean metrik-4 score per (strategy, model)')
    plt.tight_layout()
    plt.show()

for ds in ('kaiser', 'reference'):
    plot_mean_bars(ds)

## 4. Failure rate per (strategy, model)

How often each `(strategy, model)` produced an empty or unparseable
PlantUML block. Useful for spotting which combinations are unreliable.

In [ ]:
fr = (summary.groupby(['dataset', 'strategy', 'model'])['n_failed'].sum()
       / summary.groupby(['dataset', 'strategy', 'model'])['n'].sum() * 100)
fr = fr.fillna(0).reset_index(name='fail_pct')
for ds, sub in fr.groupby('dataset'):
    pivot = sub.pivot(index='strategy', columns='model', values='fail_pct').fillna(0)
    fig, ax = plt.subplots(figsize=(8, max(2, 0.4 * len(pivot))))
    im = ax.imshow(pivot.values, aspect='auto', cmap='Reds', vmin=0, vmax=100)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax.text(j, i, f'{v:.0f}%', ha='center', va='center', fontsize=8, color='black')
    ax.set_title(f'{ds} — failure rate %')
    plt.colorbar(im, ax=ax, label='%')
    plt.tight_layout()
    plt.show()

## 5. Error log

Every record that produced an error. Useful for debugging prompt
issues — `raw_excerpt` shows the first 200 chars of the LLM's output.

In [ ]:
if errors is not None and len(errors) > 0:
    print(f'{len(errors)} failure rows')
    display(errors.head(30))
else:
    print('No failures recorded.')

## 6. Cross-candidate summary

The flat `_summary.csv` — every `(strategy, model, dataset, element)`
row. This is the canonical machine-readable artefact. Useful for
spreadsheet pivot tables, statistical tests, or feeding into a
follow-up analysis notebook.

In [ ]:
summary_sorted = summary.sort_values(
    ['dataset', 'element', 'mean'], ascending=[True, True, False]
)
summary_sorted